Importação dos Dados do Kaggle

In [7]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ieee-fraud-detection')

print("Path to competition files:", path)

100%|██████████| 118M/118M [00:05<00:00, 21.4MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/ieee-fraud-detection


In [11]:
import os
conteudo = os.listdir("/root/.cache/kagglehub/competitions/ieee-fraud-detection")

print(conteudo)

['sample_submission.csv', 'test_transaction.csv', 'test_identity.csv', 'train_transaction.csv', 'train_identity.csv']


Spark

In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MinhaSessaoSpark") \
    .getOrCreate()

In [29]:
# Lê o arquivo CSV para um DataFrame do Spark
df_train = spark.read.csv("/root/.cache/kagglehub/competitions/ieee-fraud-detection/train_transaction.csv", header=True, inferSchema=True)

# Mostra as primeiras linhas do DataFrame
#df_train.show()

# Lê o arquivo CSV para um DataFrame do Spark
df_test = spark.read.csv("/root/.cache/kagglehub/competitions/ieee-fraud-detection/test_transaction.csv", header=True, inferSchema=True)

# Mostra as primeiras linhas do DataFrame
df_test.show()

+-------------+-------------+--------------+---------+-----+-----+-----+----------+-----+------+-----+-----+------+-----+-------------+-------------+-----+-----+---+---+-----+-----+---+---+-----+---+-----+---+-----+-----+-----+-----+----+-----+-----+----+----+----+----+-----+-----+----+----+----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----------------+------------------+-----------------+-----------------+------------------+-----

In [15]:
## Importar a função de Data Drift

from spark_data_drift_detector import (
    DriftConfig,
    SparkDataDriftDetector
)

In [17]:
## Definicao das Variaveis para Teste

categ = [
    'ProductCD',
    'P_emaildomain',
    'card6'
]

numerica = [
    'V96',
    'V127',
    'V133',
    'V160'
]

In [23]:
## Configurações do erro de alpha, numero de intervalos e quantidade de categorias mantidas

config = DriftConfig(
    categorical_features=categ,
    numeric_features=numerica,

    # Número de bins para variáveis numéricas
    n_bins=10,

    # Quantidade de categorias mantidas
    categorical_top_k=20,

    # Nível de significância
    relative_error=0.05
)

In [24]:
## Iniciando o Detector
detector = SparkDataDriftDetector(
    spark=spark,
    config=config
)

In [27]:
## Treinando com Base de Dados de Referencia

detector.fit(df_train)

In [30]:
## Exportar o Resultado para Analise Negocial

report = detector.compare(df_test)

report.show(
    n=100,
    truncate=False,
)

=== FEATURE METRICS ===
+-------------+------------+--------------+---------------------+---------------------+---------------------+--------------------+--------------------+---------------------+--------+--------------+--------------+----------+---------------------+-------------------+---------------------+------------------+------------------------+----+
|feature_name |feature_type|primary_metric|primary_value        |psi                  |js                   |hellinger           |ks_approx           |chi_square           |severity|severity_score|drift_detected|rank_score|missing_ref          |missing_cur        |missing_delta        |new_category_share|recommendation          |rank|
+-------------+------------+--------------+---------------------+---------------------+---------------------+--------------------+--------------------+---------------------+--------+--------------+--------------+----------+---------------------+-------------------+---------------------+---------------